# 🧭 Unity Catalog — Part 1: Learning Notebook

### Understanding Unity Catalog, one concept at a time

Welcome! 👋

This is **Part 1 of 2** in the Unity Catalog KT series:

| Notebook | Purpose |
|---|---|
| 📘 **Part 1 — Learning (this notebook)** | Understand the *concepts*, with small illustrative code snippets |
| 🧪 **Part 2 — Hands-On** | Practice building things yourself, with tasks and exercises |

This notebook focuses on **understanding**, not typing. Code snippets here are short and are
there purely to help a concept "click" — you'll get plenty of real coding practice in the
Hands-On notebook.

We'll use one running example throughout: **Microsoft** and how it manages its data.

## 🎯 Learning Objectives

By the end of this notebook, you will be able to:

1. Explain **why** companies need Unity Catalog
2. Describe the difference between **data** and **metadata**
3. Draw and explain the **Unity Catalog hierarchy**
4. Describe what happens internally when a query runs against a governed table
5. Recognize Unity Catalog objects like **Volumes**, **External Locations**, **Storage
   Credentials**, **Permissions**, **Row Filters**, and **Column Masking**

> 💡 This notebook is for KT / onboarding purposes — it is **not** a certification prep guide.

## 🤔 Before We Start... Ask Yourself One Question

> **"When a company stores 100 TB of data, how do thousands of employees access it securely?"**

Keep that question in mind. Everything in this notebook is really just one long answer to it.

## 🏢 Our Running Example: Microsoft

Let's imagine **Microsoft**. Like any large company, they have data spread across many teams:

| Team / Type      | Example Data                     |
|-------------------|-----------------------------------|
| 👩‍💼 HR            | Employee records, salaries        |
| 💰 Finance         | Budgets, revenue reports          |
| 🧑‍🤝‍🧑 Customer Data | Customer profiles, support tickets|
| 📈 Sales           | Deals, pipelines, targets         |
| 📣 Marketing       | Campaigns, ad performance         |
| 🤖 AI Models       | Trained ML models                 |
| 🖼️ Images          | Product photos, design assets     |
| 📄 PDFs            | Contracts, reports                |

**Question:** How do they manage all of this — securely, and at scale?

Let's answer this by first imagining what life looks like **without** Unity Catalog.

## 😱 Imagine There Is NO Unity Catalog

Suppose Microsoft stores everything directly in **Amazon S3**, organized into folders:

```
s3://company-data/
├── sales/
├── finance/
├── hr/
├── marketing/
└── customers/
```

Looks simple, right? It works fine... until **5000 employees join**.

Then the questions start coming in:

- ❓ Can HR access Finance?
- ❓ Can Sales modify HR data?
- ❓ Can interns delete production tables?
- ❓ Can one team see another team's salary data?
- ❓ Can we know **who deleted a table yesterday**?
- ❓ Can 3 Databricks workspaces share the same data?

**It's impossible to manage this at scale with folders alone.**

## 📌 The Real Problem Statement

Companies don't just need **storage**. They need:

- 🔐 **Governance**
- 🛡️ **Security**
- 🗂️ **Metadata**
- ✅ **Permissions**
- 📝 **Auditing**
- 🔍 **Discovery**
- 🧬 **Lineage**
- 🤝 **Sharing**

**This is exactly where Unity Catalog comes in.**

## 🧠 So... What Is Unity Catalog?

**Official definition:**

> "Unity Catalog is Databricks' unified governance solution for data and AI."

The simple version:

> 🧠 **Unity Catalog is the brain of your data platform.**

Unity Catalog tells Databricks:

- 📦 What data exists
- 📍 Where it is stored
- 👤 Who can access it
- ✅ What they can do with it
- 📝 How to track it
- 🔒 How to secure it

## ⚠️ Unity Catalog Does NOT Store Data

This is the single most important idea in this entire notebook. Read it twice. 😄

Suppose you have a table called `sales.delta`. Where does it physically live?

**Not inside Unity Catalog.**

It actually lives inside your cloud storage:

- ☁️ AWS S3, **or**
- ☁️ Azure ADLS, **or**
- ☁️ Google Cloud Storage

Unity Catalog only stores **information about** the data — things like Table Name, Owner,
Location, Permissions, Schema, Columns, Tags, Lineage.

**This "information about the data" is called `Metadata`.**

## 📦 Data vs Metadata — A Simple Way to Remember It

```
 ┌────────────────────────────┐        ┌──────────────────────────────┐
 │        Cloud Storage        │        │         Unity Catalog         │
 │   (S3 / ADLS / GCS)         │        │        (Metastore)            │
 │                              │        │                                │
 │   The actual Delta files     │        │   Table name, owner, columns  │
 │   (the DATA itself)          │        │   location, permissions, tags │
 │                              │        │            (METADATA)         │
 └────────────────────────────┘        └──────────────────────────────┘
```

**Key takeaway:**

> 🏬 **Storage stores the data. Unity Catalog governs the data.**

## 🕰️ Why Did Databricks Create Unity Catalog?

Before Unity Catalog, **every Databricks workspace had its own Hive Metastore**.

```
 Workspace A  →  Hive Metastore A
 Workspace B  →  Hive Metastore B
 Workspace C  →  Hive Metastore C
```

Each workspace had different permissions, different owners, different metadata. In short:
**chaos**. The same table could mean different things depending on which workspace you
happened to be using.

## ✅ With Unity Catalog: One Source of Truth

```
        Workspace A
              │
              ▼
Workspace B ──▶  Unity Catalog  ◀── Workspace C
              ▲
              │
        (shared governance)
```

Now **every workspace shares the same governance layer** — one place for permissions,
ownership, and auditing.

> 🎯 **One source of truth** for the entire company — no matter how many workspaces exist.

## 🛠️ What Problems Does Unity Catalog Solve?

| Category | Examples |
|---|---|
| **Identity** | Users, Groups, Service Principals |
| **Data Objects** | Catalogs, Schemas, Tables, Views, Volumes, Functions, Models |
| **Governance** | Permissions, Policies, Row Filters, Column Masks |
| **Operational** | Lineage, Audit Logs, Credentials, External Locations |
| **Organization** | Tags, Comments, Metadata |

## 🏗️ The Unity Catalog Hierarchy

```
        Metastore
            │
         Catalog
            │
         Schema
            │
          Table
```

Mapped to Microsoft:

```
        Metastore (Microsoft's single, company-wide governance layer)
            │
   ┌────────┼─────────┐
 sales   finance      hr        ← Catalogs (business domains)
   │
 gold, silver, bronze            ← Schemas (sub-groupings within a catalog)
   │
 customers, orders, leads        ← Tables (the actual structured data)
```

### 1️⃣ Metastore

> A **Metastore** is the **top-level metadata repository** in Unity Catalog. It stores
> information about all data objects (catalogs, schemas, tables, permissions, owners, etc.).
> It stores **metadata only** — never the actual data.

Think of it as the single "phone book" for an entire company's data estate. Normally, an
organization has **one Metastore per region**, shared across all its workspaces.

### 2️⃣ Catalog

> A **Catalog** is the **highest level of logical organization** inside a Metastore. It groups
> related data assets, typically based on **business domains** such as Finance, HR, or Sales.

### 3️⃣ Schema

> A **Schema** is a container within a Catalog that organizes related database objects such as
> tables, views, functions, and volumes.

### 4️⃣ Table

> A **Table** is a structured data object that stores data in rows and columns.

The **fully qualified name** of any table always follows this pattern:

```
catalog.schema.table
```

Example: `sales_catalog.gold.customers`

In [0]:
%sql
-- A quick peek: this is what asking Unity Catalog "what catalogs exist?" looks like.
-- (You'll run this for real, and build the whole hierarchy, in the Hands-On notebook.)
SHOW CATALOGS;

catalog
dev
dq_bootstrap
dq_data_5402b4d37e6e
dq_data_c4fce040f105
main
manufacturing_ford
sales_medallian
samples
streaming_analytics
system


**What just happened?**

`SHOW CATALOGS` asks Unity Catalog: *"What catalogs exist that I'm allowed to see?"* You'd
typically see built-in catalogs like `main`, `system`, and `samples`, plus any catalogs your
organization has already created.

## 🔍 INFORMATION_SCHEMA — Querying Metadata with SQL

> **`INFORMATION_SCHEMA`** is a **read-only schema** that provides SQL access to metadata
> about database objects. It does **not** store the metadata itself — it provides a way to
> **query** the metadata that the Metastore already holds.

```
      Unity Catalog Metastore
       (Actual Metadata Storage)
                 │
        INFORMATION_SCHEMA
      (Read-only SQL Interface)
                 │
           Your SQL Query
```

So remember:
- **Metastore** = actually **stores** metadata
- **INFORMATION_SCHEMA** = lets you **read** that metadata using plain SQL

In [0]:
%sql
-- One example: listing every column Unity Catalog knows about, across a catalog
SELECT * FROM sales_catalog.INFORMATION_SCHEMA.COLUMNS;

**What just happened?**

This query didn't touch any actual data — it only read **metadata** describing columns:
table name, column name, data type, position, and more.

## ⚙️ How Does Unity Catalog Work Internally?

Imagine you execute:

```sql
SELECT * FROM sales_catalog.gold.customers;
```

You might assume Spark immediately reads the table. **That's wrong.** Let's trace what
actually happens.

```
 You
  │  SELECT * FROM sales_catalog.gold.customers;
  ▼
 Spark
  │  "Does this table exist?"
  ▼
 Unity Catalog  ──▶  checks its metadata
  │
  ├── Not found?  ───────────────▶  ❌ Error
  │
  ▼  Found!
 "Who are you?"  ──▶  e.g. Gokul
  │
  ▼
 "Do you have SELECT permission?"
  │
  ├── No  ───────────────────────▶  ❌ Permission denied
  │
  ▼  Yes
 "Any row filters?"  ──▶  e.g. department = 'HR'
  │
  ▼
 "Any column masks?"  ──▶  e.g. salary column → ******
  │
  ▼
 "Where is the table physically stored?"  ──▶  s3://company-data/gold/customers
  │
  ▼
 "Which credential should I use to access that path?"  ──▶  IAM Role
  │
  ▼
 Spark reads the Delta files
  │
  ▼
 ✅ Results are returned to you
```

⏱️ **This entire process takes milliseconds** — but it shows exactly why Unity Catalog sits
in the **critical path** for governance on every single query.

### What Exactly *Is* a Metastore?

A common point of confusion for beginners: **a Metastore is not a Catalog.**

```
Unity Catalog  =>  Metastore  =>  Stores Metadata
```

Every Databricks account typically has **one Metastore per region**, and every workspace in
that region attaches to it — giving us the "one source of truth" we talked about earlier.

## 🧩 Quick Introduction to Other Unity Catalog Objects

So far we've focused on the core hierarchy (Metastore → Catalog → Schema → Table). Here are
a few more objects you'll run into on real projects. You'll get to actually **build** each of
these in the Hands-On notebook — for now, just focus on understanding what each one is *for*.

### 📁 Volume

> A **Volume** is a governed storage object used to store and manage **unstructured files**
> such as PDFs, images, CSVs, Excel files, JSON files, machine learning models, and other
> documents.

Remember Microsoft's **Images, PDFs, and AI Models**? Volumes are how Unity Catalog governs
access to files like these — the same way tables govern access to structured rows and
columns.

In [0]:
%sql
-- Just the shape of it — a volume always lives at a predictable path:
-- /Volumes/<catalog>/<schema>/<volume_name>/
LIST '/Volumes/sales_catalog/gold/contracts_volume';

### 🔑 Storage Credential

> A **Storage Credential** is a secure object that stores **cloud authentication details**
> (such as an AWS IAM Role, Azure Managed Identity, or GCP Service Account) used by Unity
> Catalog to access cloud storage **without exposing credentials to users**.

This is why, in our "how a query runs internally" walkthrough, Unity Catalog could say *"use
this IAM Role"* — without you ever seeing or handling the actual cloud secret.

> ⚠️ Creating one requires **Metastore Admin** privileges — usually a one-time platform-team
> setup, not something every engineer does day to day.

In [0]:
%sql
-- Creating a managed volume to store unstructured files
CREATE VOLUME IF NOT EXISTS sales_catalog.gold.contracts_volume
COMMENT 'Volume for storing customer contracts and legal documents';

**What just happened?**

This creates a governed storage location at `/Volumes/sales_catalog/gold/contracts_volume/` where you can upload PDFs, images, models, or any files — with Unity Catalog tracking who can read, write, or manage them.

In [0]:
%sql
-- Creating a storage credential (requires Metastore Admin privileges)
CREATE STORAGE CREDENTIAL IF NOT EXISTS aws_sales_credential
WITH (AWS_IAM_ROLE 'arn:aws:iam::123456789012:role/unity-catalog-sales-role')
COMMENT 'Credential for accessing sales data in S3';

**What just happened?**

This securely registers an AWS IAM Role with Unity Catalog. Users never see the actual credentials — Unity Catalog uses them behind the scenes when accessing S3 paths.

In [0]:
%sql
-- Creating an external location pointing to S3
CREATE EXTERNAL LOCATION IF NOT EXISTS sales_data_lake
URL 's3://company-sales-bucket/gold/'
WITH (STORAGE CREDENTIAL aws_sales_credential)
COMMENT 'Production sales data location';

**What just happened?**

We've mapped a specific S3 path to a storage credential. Now Unity Catalog can govern access to this location — you can create external tables pointing here, and UC will use the credential automatically.

In [0]:
%sql
-- Creating a managed table (Unity Catalog manages the data)
CREATE TABLE IF NOT EXISTS sales_catalog.gold.customers (
  customer_id BIGINT,
  name STRING,
  email STRING,
  department STRING,
  salary DECIMAL(10,2),
  created_at TIMESTAMP
)
COMMENT 'Customer master data';

-- Or create an external table (you manage the data location)
-- CREATE EXTERNAL TABLE sales_catalog.gold.customers
-- LOCATION 's3://company-sales-bucket/gold/customers/';
-- This requires the external location to be already created and accessible

**What just happened?**

* **Managed table**: Unity Catalog owns both metadata and data lifecycle. When you drop the table, the data is deleted.
* **External table**: You own the data in cloud storage; Unity Catalog just governs access. Dropping the table doesn't delete the underlying files.

### 🔐 Privileges

> **Privileges** are the specific **actions** (like `SELECT`, `MODIFY`, `CREATE TABLE`, `USE CATALOG`) that permissions allow or deny. When you `GRANT` a privilege, you're saying *"this user/group can perform this action on this object."*

Common privileges:

| Privilege | What It Allows |
|---|---|
| `USE CATALOG` | Browse and see the catalog |
| `USE SCHEMA` | Browse and see the schema |
| `SELECT` | Read data from a table/view |
| `MODIFY` | Insert, update, delete rows |
| `CREATE TABLE` | Create tables in a schema |
| `ALL PRIVILEGES` | All available privileges on an object |

In [0]:
%sql
-- See what privileges are granted on a table
SHOW GRANTS ON TABLE sales_catalog.gold.customers;

-- See what privileges a user/group has
-- SHOW GRANTS ON CATALOG sales_catalog TO `finance_team`;

**What just happened?**

`SHOW GRANTS` reveals exactly who can do what — making governance transparent and auditable. You can inspect privileges at any level: metastore, catalog, schema, table, or even for a specific principal.

In [0]:
%sql
-- Step 1: Create a function that defines the filter logic
CREATE OR REPLACE FUNCTION sales_catalog.gold.hr_filter(department STRING)
RETURN IF(IS_ACCOUNT_GROUP_MEMBER('hr_team'), TRUE, department = 'HR');

-- Step 2: Apply the row filter to a table
ALTER TABLE sales_catalog.gold.customers
SET ROW FILTER sales_catalog.gold.hr_filter ON (department);

-- Now when someone queries this table:
-- - HR team members see ALL rows
-- - Everyone else ONLY sees rows where department = 'HR'

**What just happened?**

Unity Catalog now automatically applies this filter **every time** anyone queries the table — users don't write `WHERE` clauses, and they can't bypass it. This is true row-level security.

In [0]:
%sql
-- Step 1: Create a function that defines the mask logic
CREATE OR REPLACE FUNCTION sales_catalog.gold.salary_mask(salary DECIMAL(10,2))
RETURN IF(IS_ACCOUNT_GROUP_MEMBER('finance_team'), salary, NULL);

-- Step 2: Apply the column mask to a specific column
ALTER TABLE sales_catalog.gold.customers
ALTER COLUMN salary SET MASK sales_catalog.gold.salary_mask;

-- Now when someone queries SELECT * FROM customers:
-- - Finance team members see the actual salary values
-- - Everyone else sees NULL in the salary column

**What just happened?**

The `salary` column is now protected. Non-finance users still see the column in query results, but its value is masked. You could also return `'REDACTED'`, `0`, or any other value instead of `NULL`.

### ✅ Permissions

> **Permissions** are access control rules that determine which **users, groups, or service
> principals** can perform actions (such as `SELECT`, `MODIFY`, `CREATE`, or `DELETE`) on
> Unity Catalog objects.

In [0]:
%sql
-- The general shape of a permission grant:
GRANT SELECT ON TABLE sales_catalog.gold.customers TO `finance_team`;

**What just happened?**

`GRANT` (and its opposite, `REVOKE`) is how you actually answer questions like *"Can Sales
modify HR data?"* — explicitly, auditably, and without touching a single storage folder
permission.

## 📚 Summary

- Microsoft has huge amounts of data (HR, Finance, Sales, Marketing, AI Models, Images, PDFs).
- Without Unity Catalog, managing access for 5000+ employees across raw storage folders is
  **impossible to do safely**.
- **Unity Catalog is the brain of the data platform** — it doesn't store data, it **governs**
  it.
- Data lives in cloud storage (S3 / ADLS / GCS). **Metadata** lives in the **Unity Catalog
  Metastore**.
- The hierarchy is: **Metastore → Catalog → Schema → Table**.
- `INFORMATION_SCHEMA` lets you query that metadata using plain SQL.
- Every query passes through a governance checkpoint: *table exists → who are you → do you
  have permission → row filters → column masks → where's the data → which credential → read
  → return results.*
- Beyond tables, Unity Catalog also governs **Volumes**, **External Locations**, **Storage
  Credentials**, **Permissions**, **Row Filters**, and **Column Masking**.

### 🧠 Key Takeaways to Remember

> 1. **Unity Catalog is the brain of your data platform.**
> 2. **Storage stores the data. Unity Catalog governs the data.**
> 3. **Metastore → Catalog → Schema → Table.**
> 4. **Every query passes through a governance checkpoint — every single time.**

## ✅ Knowledge Check

Try answering these before checking the solutions below — no peeking! 😉

1. Does Unity Catalog store actual data files? Why or why not?
2. What is the correct order of the Unity Catalog hierarchy?
3. What is the difference between a Metastore and `INFORMATION_SCHEMA`?
4. In the "how a query runs internally" flow, what happens right after Unity Catalog checks
   *"who are you"*?
5. Which Unity Catalog object would you use to govern access to a folder of PDF contracts?
6. What is the purpose of a Storage Credential?

### 🔓 Answers

1. **No.** Unity Catalog only stores metadata — information *about* the data (name, owner,
   location, schema, permissions). The actual data lives in cloud storage.
2. **Metastore → Catalog → Schema → Table.**
3. The **Metastore** actually *stores* the metadata. **`INFORMATION_SCHEMA`** is a read-only
   SQL interface used to *query* that stored metadata — it doesn't store anything itself.
4. Unity Catalog checks **whether you have the required permission** (e.g. `SELECT`) —
   if not, you get a permission denied error.
5. A **Volume** — Volumes are designed specifically to govern unstructured files like PDFs,
   images, and models.
6. A **Storage Credential** securely stores cloud authentication details (like an IAM Role)
   so Unity Catalog can access cloud storage on your behalf, without exposing secrets to
   users.

## ➡️ What's Next?

You now understand **what** Unity Catalog is, **why** it exists, and **how** it works
internally.

Head over to the **Part 2 — Hands-On Notebook** to actually build a catalog, schema, and
table hierarchy, write your own permission grants, and create your own row filters and
column masks. 🚀